# Compare training runs & per-tile AP/AR heatmaps

**Prefer no Jupyter?** From the project root run:

```bash
python BoulderCalculator/experiments/eval/run_compare_report.py
```

This notebook is optional. Use the **`.venv_boulder`** kernel. Path discovery walks upward to find `BoulderCalculator/scripts/eval_utils.py`.

Companions: `eval_compare_runs.py`, `eval_per_tile.py`. See `experiments/eval/README.md` for laptop paths (`coco_geo_*_rgb_dsm` is often an empty stub).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Walk up until BoulderCalculator/scripts/eval_utils.py exists (cwd-safe).
ROOT = Path.cwd().resolve()
for cand in [ROOT, *ROOT.parents]:
    if (cand / "BoulderCalculator" / "scripts" / "eval_utils.py").is_file():
        ROOT = cand
        break
else:
    raise FileNotFoundError(
        "Could not find project root. Open from tamucc/ or experiments/eval/ "
        "and select the .venv_boulder kernel.\n"
        "Or skip Jupyter: python BoulderCalculator/experiments/eval/run_compare_report.py"
    )

SCRIPTS = ROOT / "BoulderCalculator" / "scripts"
sys.path.insert(0, str(SCRIPTS))

from eval_utils import (
    compare_metrics_valid,
    discover_geo_runs,
    extract_eval_curve,
    load_metrics_jsonl,
    load_split_yaml,
    plot_learning_curves,
    plot_tile_heatmaps,
    summarize_by_split_membership,
)

SEG = ROOT / "segmentation"
GEO_SPLITS = ROOT / "BoulderCalculator" / "experiments" / "geo_splits"
print("ROOT", ROOT)
print("SCRIPTS", SCRIPTS, "exists=", (SCRIPTS / "eval_utils.py").is_file())
print("geo runs", list(discover_geo_runs(SEG).keys()))

## 1. Quick comparison of `metrics_valid.json`

Auto-discovers `segmentation/training_run_geo_*/metrics_valid.json`. Edit `runs` to add any other training dirs.

In [ ]:
runs = discover_geo_runs(SEG, prefix="training_run_geo_")
df = compare_metrics_valid(runs)
show_cols = [c for c in df.columns if c.startswith(("bbox/", "segm/"))]
display(df[show_cols].sort_values("bbox/AP50", ascending=False).round(2))

print("Best bbox/AP50:", df["bbox/AP50"].idxmax(), f"{df['bbox/AP50'].max():.2f}")
print("Best segm/AP50:", df["segm/AP50"].idxmax(), f"{df['segm/AP50'].max():.2f}")

## 2. Validation vs iteration (saturation)

Reads each run’s `metrics.json` periodic eval events.

In [ ]:
curves = {}
for name, valid_path in runs.items():
    mpath = Path(valid_path).parent / "metrics.json"
    if mpath.exists():
        curves[name] = extract_eval_curve(load_metrics_jsonl(mpath))

for metric in ("bbox/AP50", "segm/AP50", "bbox/AR100", "segm/AR100"):
    if not any(metric in p for curve in curves.values() for p in curve):
        continue
    fig = plot_learning_curves(curves, metric=metric)
    plt.show()

## 3. Per-tile AP / AR heatmaps

On laptops where `coco_geo_baseline_rgb_dsm` is an empty stub:

```bash
python BoulderCalculator/scripts/eval_per_tile.py \
  --gt-json segmentation/coco_geo_baseline/testing_annotations.json \
  --image-dir segmentation/tiling_rgb_dsm_24 \
  --image-dir segmentation/tiling_rgb_dsm_25 \
  --model segmentation/training_run_geo_baseline/model_final.pth \
  --four-band --device cuda \
  --output-dir segmentation/eval_per_tile_baseline_test \
  --split-config BoulderCalculator/experiments/geo_splits/baseline.yaml \
  --split-config BoulderCalculator/experiments/geo_splits/blocks_alt_a.yaml \
  --split-config BoulderCalculator/experiments/geo_splits/blocks_alt_b.yaml
```

Add `--merge-iou 0.4` for sliding-window **with merging**. Then load the CSV below.

In [ ]:
per_tile_csv = SEG / "eval_per_tile_baseline_test" / "per_tile_metrics.csv"
if per_tile_csv.exists():
    tiles = pd.read_csv(per_tile_csv)
    print(tiles[["stem", "year", "row", "col", "coco_AP50", "coco_AR100", "precision", "recall", "gt_count"]].head(20))
    print("\nMeans:")
    print(tiles[["coco_AP50", "coco_AR100", "precision", "recall"]].mean(numeric_only=True))
    rows = tiles.to_dict(orient="records")
    for metric, fig in plot_tile_heatmaps(rows, metrics=("coco_AP50", "coco_AR100", "recall", "precision")):
        plt.show()
else:
    print(f"No {per_tile_csv} yet — run eval_per_tile.py first (see markdown above).")

## 4. Are baseline test tiles easier?

After you have **one** per-tile metrics table on a common pool, average each geo-setup’s **test** membership. If baseline’s mean stays high under the *same* predictions, the hold-out tiles are likely easier.

In [ ]:
if per_tile_csv.exists():
    rows = pd.read_csv(per_tile_csv).to_dict(orient="records")
    setups = [
        "baseline.yaml",
        "blocks_alt_a.yaml",
        "blocks_alt_b.yaml",
        "north_south.yaml",
        "sporadic_aligned.yaml",
    ]
    summary = []
    for name in setups:
        path = GEO_SPLITS / name
        if not path.exists():
            continue
        cfg = load_split_yaml(path)
        ap = summarize_by_split_membership(rows, cfg, "coco_AP50", "test")
        ar = summarize_by_split_membership(rows, cfg, "coco_AR100", "test")
        rec = summarize_by_split_membership(rows, cfg, "recall", "test")
        summary.append({
            "setup": cfg.get("id", path.stem),
            "n_test_tiles_scored": ap["n"],
            "mean_AP50": ap["mean"],
            "mean_AR100": ar["mean"],
            "mean_recall": rec["mean"],
        })
    display(pd.DataFrame(summary).sort_values("mean_AP50", ascending=False).round(2))
else:
    print("Run per-tile eval first.")

## 5. Sliding window: with vs without merging

Run `eval_per_tile.py` twice — with and without `--merge-iou 0.4` — then compare means here.

In [ ]:
raw = SEG / "eval_per_tile_baseline_test_raw" / "per_tile_metrics.csv"
merged = SEG / "eval_per_tile_baseline_test_merged" / "per_tile_metrics.csv"
if raw.exists() and merged.exists():
    a, b = pd.read_csv(raw), pd.read_csv(merged)
    cols = ["coco_AP50", "coco_AR100", "precision", "recall", "pred_count"]
    print("RAW means:\n", a[cols].mean(numeric_only=True).round(3))
    print("\nMERGED means:\n", b[cols].mean(numeric_only=True).round(3))
else:
    print("Optional: produce raw/ and merged/ outputs with eval_per_tile.py to compare here.")